In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# 1. Kiểm tra GPU, đồng bộ code mới nhất vào /content/EEG-mini-project và cài xgboost
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_GIT_URL = "https://github.com/Longkyoladn/EEG-mini-project.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = Path("/content/EEG-mini-project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/EEG-mini-project")
LOCAL_FEATURES_DIR = Path("/content/features")

subprocess.run(["nvidia-smi"], check=False)

if not COLAB_REPO_DIR.exists():
    if (DRIVE_PROJECT_DIR / "src").is_dir() and (DRIVE_PROJECT_DIR / "splits").is_dir():
        print(f"Copying repo code from Drive {DRIVE_PROJECT_DIR} -> {COLAB_REPO_DIR} ...")
        shutil.copytree(
            DRIVE_PROJECT_DIR,
            COLAB_REPO_DIR,
            ignore=shutil.ignore_patterns("features", "results", "data", "oddball_data", ".git"),
        )
    else:
        subprocess.run(
            ["git", "clone", "-b", REPO_BRANCH, REPO_GIT_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
elif (COLAB_REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull"], check=False)

os.chdir(COLAB_REPO_DIR)
if str(COLAB_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(COLAB_REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0.0"], check=True)
print(f"Ready at: {Path.cwd()}")

Ready at: /content/EEG-mini-project


In [3]:
# 2. Kiểm tra Google Drive & copy features/temporal + features/wavelet sang /content/features
# (Các file đã copy sẵn trong /content/features sẽ được giữ nguyên, không copy lại)
from scripts.training.ML3.train_ml3 import main as train_ml3_main, setup_colab_drive_paths

paths = setup_colab_drive_paths(
    drive_project_dir=DRIVE_PROJECT_DIR,
    stage_to_content=True,
    local_features_dir=LOCAL_FEATURES_DIR,
)
for k, v in paths.items():
    print(f"{k}: {v}")

[Stage Features] /content/drive/MyDrive/EEG-mini-project/features -> /content/features | copied=0, reused=78, transferred=0.0 MB
project_dir: /content/drive/MyDrive/EEG-mini-project
drive_features_dir: /content/drive/MyDrive/EEG-mini-project/features
features_dir: /content/features
temporal_dir: /content/features/temporal
wavelet_dir: /content/features/wavelet
output_dir: /content/drive/MyDrive/EEG-mini-project/results/ml3


In [ ]:
# 3. Kiểm tra nhanh cấu hình & số lượng feature (--plan-only)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--plan-only",
    ]
)

In [ ]:
# 4. Chạy Cross-Subject 5-Fold (5 kịch bản chính)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--experiments", "temporal", "wavelet_w5", "wavelet_w8", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

In [ ]:
# 5. Chạy LOSO (3 kịch bản chính: temporal, combined_w5, combined_w8)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "loso",
        "--experiments", "temporal", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)